In [1]:
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

print("✓ Core mathematical libraries successfully loaded.")

✓ Core mathematical libraries successfully loaded.


In [2]:
# FIX 1: Updated directory paths to point to your new NeuralProt_Beta workspace
PROCESSED_DIR   = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/processed_data"
MODEL_DIR       = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/models"
ASSIGNMENT_PATH = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/go_group_assignment_v3.json"
 
# Sweep settings: test 100 fine decision lines between 0.05 and 0.95
THRESHOLDS = np.linspace(0.05, 0.95, 100)
BATCH_SIZE = 64  # Batch size can be larger here since we are not calculating training slopes
 
# FIX 2: Dynamic Detection. We read the v3 blueprint to find all 370+ active folders automatically!
with open(ASSIGNMENT_PATH, "r") as f:
    _blueprint_data = json.load(f)
TUNE_GROUPS = list(_blueprint_data["groups"].keys())
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computer Processing Unit : {DEVICE}")
print(f"Sweeping {len(THRESHOLDS)} threshold settings per group.")
print(f"Total dynamic folders scheduled for optimization: {len(TUNE_GROUPS)}")

Computer Processing Unit : cpu
Sweeping 100 threshold settings per group.
Total dynamic folders scheduled for optimization: 375


In [3]:
class ProteinMLP(nn.Module):
    # FIX 3: Changed the default input layer size from 428 to exactly 498!
    def __init__(self, input_dim=498, num_classes=128, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
 
            nn.Linear(256, num_classes),
        )
 
    def forward(self, x):
        return self.network(x)
 
print("✓ ProteinMLP brain architecture updated to 498 inputs.")

✓ ProteinMLP brain architecture updated to 498 inputs.


In [4]:
class ProteinDataset(torch.utils.data.Dataset):
    def __init__(self, indices, feature_matrix, label_matrix):
        self.indices        = indices
        self.feature_matrix = feature_matrix
        self.label_matrix   = label_matrix
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, idx):
        i        = self.indices[idx]
        features = torch.tensor(self.feature_matrix[i], dtype=torch.float32)
        label    = torch.tensor(self.label_matrix[i],   dtype=torch.float32)
        return features, label
 
print("✓ PyTorch Dataset wrapper class verified.")

✓ PyTorch Dataset wrapper class verified.


In [5]:
def get_val_probabilities(model, val_loader, device):
    """Feeds validation proteins through the network to collect raw guess numbers."""
    model.eval()
    all_probs   = []
    all_targets = []
 
    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            logits   = model(features)
            probs    = torch.sigmoid(logits)  # Convert logits to probability percentages
            all_probs.append(probs.cpu().numpy())
            all_targets.append(labels.numpy())
 
    return np.vstack(all_probs), np.vstack(all_targets)
 
 
def find_best_threshold(probs, targets, thresholds):
    """Sweeps through the 100 test lines to calculate which one scores the highest F1."""
    results = []
 
    for thresh in thresholds:
        preds     = (probs >= thresh).astype(float)
        f1        = f1_score(targets,  preds, average="macro", zero_division=0)
        precision = precision_score(targets, preds, average="macro", zero_division=0)
        recall    = recall_score(targets,    preds, average="macro", zero_division=0)
        results.append({
            "threshold": round(float(thresh), 4),
            "f1":        round(float(f1), 6),
            "precision": round(float(precision), 6),
            "recall":    round(float(recall), 6),
        })
 
    # Sort from highest score to lowest score and return the top option
    results.sort(key=lambda x: -x["f1"])
    return results[0], results
 
 
def tune_group(group_name, processed_dir, model_dir, thresholds, device, batch_size):
    """Loads a folder box, recreates the exact split, and calculates the sweet spot."""
    # FIX 4: Implemented colon and slash protection so paths match your flat folders perfectly
    safe_name = group_name.replace(":", "_").replace("/", "_")
    
    group_dir  = os.path.join(processed_dir, safe_name)
    model_path = os.path.join(model_dir, f"{safe_name}_best.pt")
 
    if not os.path.exists(model_path):
        print(f"   ✗ Skipping: Model weight file not found at {model_path}")
        return None
 
    # Load metadata text configurations
    with open(os.path.join(group_dir, "accessions.json")) as f:
        accessions = json.load(f)
 
    # FIX 5: Updated to read zipped 'labels.npz' and clean 'terms.json' files
    with np.load(os.path.join(group_dir, "labels.npz")) as archive:
        label_matrix = archive["labels"]
        
    feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
 
    with open(os.path.join(group_dir, "terms.json")) as f:
        go_term_list = json.load(f)
 
    n_classes = len(go_term_list)
 
    # Load the exact same test split that was saved during training
    test_idx_path = os.path.join(group_dir, "test_idx.json")
    with open(test_idx_path) as f:
        test_idx = json.load(f)

#    Reconstruct val_idx from everything that is NOT in test
    test_idx_set = set(test_idx)
    remaining = [i for i in range(len(accessions)) if i not in test_idx_set]
    _, val_idx = train_test_split(remaining, test_size=0.111, random_state=42)

    val_dataset = ProteinDataset(val_idx, feature_matrix, label_matrix)
    val_loader  = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
 
    # Load model weights safely
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model      = ProteinMLP(num_classes=n_classes).to(device)
    model.load_state_dict(checkpoint["model_state"])
 
    trained_f1 = checkpoint.get("val_f1", None)
 
    # Extract prediction matrices
    probs, targets = get_val_probabilities(model, val_loader, device)
 
    # Find winning threshold value
    best, all_results = find_best_threshold(probs, targets, thresholds)
 
    # Calculate performance baseline at standard 0.50 threshold for gain checking
    default_preds = (probs >= 0.5).astype(float)
    default_f1    = f1_score(targets, default_preds, average="macro", zero_division=0)
    default_p     = precision_score(targets, default_preds, average="macro", zero_division=0)
    default_r     = recall_score(targets, default_preds, average="macro", zero_division=0)
 
    return {
        "group_name":      group_name,
        "n_go_terms":      n_classes,
        "n_val_proteins":  len(val_idx),
        "trained_val_f1":  round(float(trained_f1), 6) if trained_f1 else None,
        "default_threshold": {
            "threshold": 0.5,
            "f1":        round(float(default_f1), 6),
            "precision": round(float(default_p), 6),
            "recall":    round(float(default_r), 6),
        },
        "optimal_threshold": best,
        "f1_gain":           round(best["f1"] - float(default_f1), 6),
        "all_thresholds":    all_results,
    }
 
print("✓ Tuning pipeline math functions successfully compiled.")

✓ Tuning pipeline math functions successfully compiled.


In [6]:
print("Launching threshold sweep optimization pipeline...")
print("=" * 75)
 
all_results = {}
 
for group_name in TUNE_GROUPS:
    safe_name = group_name.replace(":", "_").replace("/", "_")
    result = tune_group(
        group_name   = group_name,
        processed_dir = PROCESSED_DIR,
        model_dir    = MODEL_DIR,
        thresholds   = THRESHOLDS,
        device       = DEVICE,
        batch_size   = BATCH_SIZE,
    )
 
    if result is None:
        continue
 
    all_results[group_name] = result
 
    d = result["default_threshold"]
    o = result["optimal_threshold"]
 
    print(f"  [TUNE COMPLETE] Folder: {safe_name}")
    print(f"     Jobs: {result['n_go_terms']} | Validation Samples: {result['n_val_proteins']:,}")
    print(f"     Standard (0.50) -> F1: {d['f1']:.4f} | P: {d['precision']:.4f} | R: {d['recall']:.4f}")
    print(f"     Custom ({o['threshold']:.2f})   -> F1: {o['f1']:.4f} | P: {o['precision']:.4f} | R: {o['recall']:.4f}")
    print(f"     Score Performance Gain -> +{result['f1_gain']:.4f}")
    print("-" * 75)

Launching threshold sweep optimization pipeline...
  [TUNE COMPLETE] Folder: positive_regulation_of_DNA-templated_transcription_GO_0045893
     Jobs: 16 | Validation Samples: 480
     Standard (0.50) -> F1: 0.6702 | P: 0.5349 | R: 0.9784
     Custom (0.94)   -> F1: 0.8282 | P: 0.8646 | R: 0.8417
     Score Performance Gain -> +0.1580
---------------------------------------------------------------------------
  [TUNE COMPLETE] Folder: regulation_of_mRNA_stability_GO_0043488
     Jobs: 16 | Validation Samples: 47
     Standard (0.50) -> F1: 0.7729 | P: 0.7565 | R: 0.8021
     Custom (0.86)   -> F1: 0.7889 | P: 0.8040 | R: 0.7773
     Score Performance Gain -> +0.0161
---------------------------------------------------------------------------
  [TUNE COMPLETE] Folder: heterochromatin_formation_GO_0031507
     Jobs: 16 | Validation Samples: 61
     Standard (0.50) -> F1: 0.7183 | P: 0.6763 | R: 0.7912
     Custom (0.66)   -> F1: 0.7492 | P: 0.7265 | R: 0.7912
     Score Performance Gain ->

In [7]:
"""
================================================================================
NeuralProt — Master Threshold Summary & JSON Exporter (Block 7)
================================================================================
What this code does:
1. It gathers all the threshold testing answers from the computer's memory.
2. It sorts them from the LOWEST score to the HIGHEST score so you can see the 
   weakest groups immediately.
3. It opens each group's metadata file to grab the exact total protein counts.
4. It prints a beautiful, aligned report card grid table.
5. It saves a clean setup file called 'threshold_results.json' on your disk.
"""

print("\n" + "=" * 92)
print("              MASTER THRESHOLD TUNING REPORT CARD (WORST PERFORMERS FIRST)")
print("=" * 92)
print(f"  {'Model Group Folder Name':<35} | {'Proteins':<8} | {'GO Terms':<8} | {'Old F1':<7} | {'New F1':<7} | {'Sweet Spot':<10}")
print("-" * 92)

# SORTING ENGINE: Sorts the dictionary items by the new optimized F1 score 
# in ascending order (lowest numbers come first).
sorted_results = sorted(all_results.items(), key=lambda x: x[1]["optimal_threshold"]["f1"])

for group_name, result in sorted_results:
    # Use our safe name cleaner so the script looks in the right folders
    safe_name = group_name.replace(":", "_").replace("/", "_")
    
    d = result["default_threshold"]
    o = result["optimal_threshold"]
    
    # Mentor Trick: Read the true total protein count straight from the metadata file
    total_proteins = "Unknown"
    metadata_path = os.path.join(PROCESSED_DIR, safe_name, "metadata.json")
    
    if os.path.exists(metadata_path):
        try:
            with open(metadata_path, "r") as f:
                meta = json.load(f)
                # Format with commas (e.g., 23,517)
                total_proteins = f"{meta.get('n_proteins', 'Unknown'):,}"
        except Exception:
            total_proteins = "Error"

    # Truncate text names longer than 35 letters so the table columns stay perfectly straight
    short_name = safe_name if len(safe_name) <= 35 else safe_name[:32] + "..."
    
    # Print the clean data row line
    print(f"  {short_name:<35} | {total_proteins:<8} | {result['n_go_terms']:<8} | {d['f1']:<7.4f} | {o['f1']:<7.4f} | {o['threshold']:<10.2f}")

# ── SAVING THE SETTINGS CHECKLIST TO THE HARD DRIVE ───────────────────────────
output_path = os.path.join(MODEL_DIR, "threshold_results.json")

# Clean out the massive list of raw graph plot numbers to save disk space
save_results = {}
for group_name, result in all_results.items():
    save_results[group_name] = {k: v for k, v in result.items() if k != "all_thresholds"}
 
with open(output_path, "w") as f:
    json.dump(save_results, f, indent=2)
 
print("=" * 92)
print(f"✓ Success! Optimal threshold maps written to disk -> {output_path}")
print("  Your live web app backend can now use these custom sweet spots automatically.")
print("=" * 92)


              MASTER THRESHOLD TUNING REPORT CARD (WORST PERFORMERS FIRST)
  Model Group Folder Name             | Proteins | GO Terms | Old F1  | New F1  | Sweet Spot
--------------------------------------------------------------------------------------------
  Fallback_cellular_component_8       | 24,971   | 50       | 0.0816  | 0.1584  | 0.94      
  intracellular_membrane-bounded_o... | 32,388   | 38       | 0.2151  | 0.3595  | 0.94      
  membrane_GO_0016020                 | 26,266   | 35       | 0.2058  | 0.4104  | 0.94      
  Fallback_molecular_function_1       | 7,888    | 50       | 0.1875  | 0.4323  | 0.95      
  response_to_other_organism_GO_00... | 5,018    | 26       | 0.3392  | 0.4482  | 0.82      
  cytoplasm_GO_0005737                | 23,517   | 15       | 0.2165  | 0.4759  | 0.95      
  intracellular_membraneless_organ... | 11,865   | 74       | 0.2577  | 0.4817  | 0.95      
  G_protein-coupled_receptor_signa... | 2,937    | 36       | 0.3228  | 0.5398  | 0.87 